In [1]:
import random, numpy as np, torch
MAX_LEN = 256
SEED = 2024
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
import torch, psutil

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))

GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.2
RAM (GB): 179.4


In [3]:
import torch
import pandas as pd
import numpy as np

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import pandas as pd
from transformers import (
	RobertaTokenizer,
	RobertaForSequenceClassification,
	Trainer,
	TrainingArguments
)


In [4]:
DEV_PATH  = "/content/development_processed.csv"
EVAL_PATH = "/content/evaluation_processed.csv"

df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
	df["title"]   = df["title"].fillna("").astype(str)
	df["article"] = df["article"].fillna("").astype(str)


In [5]:
def build_text(df):
	return df["title"] + "\n\n" + df["article"]

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

In [6]:
# ============================================================
class NewsDataset(torch.utils.data.Dataset):
	def __init__(self, df, tokenizer, with_labels=True, max_length=MAX_LEN):
		self.texts  = df["text"].tolist()
		self.labels = df["label"].values if with_labels else None
		self.tokenizer = tokenizer
		self.max_length = max_length

	def __len__(self):
		return len(self.texts)

	def __getitem__(self, idx):
		enc = self.tokenizer(
			self.texts[idx],
			truncation=True,
			padding="max_length",
			max_length=self.max_length,
			return_tensors="pt"
		)

		item = {k: v.squeeze(0) for k, v in enc.items()}

		if self.labels is not None:
			item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

		return item

In [7]:
MODEL_NAME = "roberta-large"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

model = RobertaForSequenceClassification.from_pretrained(
	MODEL_NAME,
	num_labels=7
).cuda()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
train_ds = NewsDataset(df_dev, tokenizer, with_labels=True)
eval_ds  = NewsDataset(df_eval, tokenizer, with_labels=False)



In [9]:
args = TrainingArguments(
	output_dir=f"./roberta_seed{SEED}",
	learning_rate=2e-5,
	per_device_train_batch_size=8,
	gradient_accumulation_steps=4,
	num_train_epochs=3,
	fp16=True,

	eval_strategy="no",
	save_strategy="no",

	seed=SEED,
	data_seed=SEED,

	logging_steps=100,
	report_to="none"
)


In [10]:
trainer = Trainer(
	model=model,
	args=args,
	train_dataset=train_ds
)

trainer.train()


Step,Training Loss
100,1.234200
200,0.901300
300,0.840800
400,0.779400
500,0.756800
600,0.748800
700,0.750900
800,0.738100
900,0.724300
1000,0.767000


TrainOutput(global_step=7500, training_loss=0.5878181849161784, metrics={'train_runtime': 2591.1843, 'train_samples_per_second': 92.618, 'train_steps_per_second': 2.894, 'total_flos': 1.1182945911737702e+17, 'train_loss': 0.5878181849161784, 'epoch': 3.0})

In [11]:
pred_out = trainer.predict(eval_ds)
logits = pred_out.predictions

np.save(f"logits_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.npy", logits)

preds = logits.argmax(axis=1)

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": preds.astype(int)
})

submission.to_csv(f"submission_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.csv", index=False)

print(f"Saved submission_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.csv")
print(f"Saved logits_roberta_processed_seed{SEED}_MAXLEN{MAX_LEN}.npy")

Saved submission_roberta_processed_seed2024_MAXLEN256.csv
Saved logits_roberta_processed_seed2024_MAXLEN256.npy
